[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/049.%20The%20Causal%20%26%20Regression-Based%20Forecasting%20Problem/P49-Tier-3_executed.ipynb)

---

# 49. The Causal and Regression-Based Forecasting Problem

## Tier 3: Genetic Algorithm for Multivariate Model Selection

### Purpose and scope

This notebook uses a genetic algorithm (GA) to search for a subset of predictors and tune the parameters of an Elastic Net regression model. The example contains 12 candidate predictors and a synthetic port-throughput response. A chromosome encodes both a binary feature-selection decision for each predictor and three real-valued parameter genes. Candidate chromosomes are evaluated by five-fold cross-validation, with a feature-count penalty added to the mean squared error (MSE).

The notebook's outputs are a selected feature subset, the associated Elastic Net parameters, the GA's fitness history, and diagnostic summaries and plots. The GA is a stochastic search procedure: it explores candidate solutions but does not guarantee a globally optimal solution.

### Modeling assumptions and limitations

- The data are synthetic and generated with a fixed NumPy random seed. Six predictors are designated as signal-generating variables, while six are generated as noise variables. These labels describe how this example was constructed; selecting a variable does not establish a causal relationship in real data.
- The response-generation equation includes an interaction between trade volume and GDP growth, but the candidate predictors contain only the original variables, not their product. The fitted Elastic Net therefore models additive main effects and cannot represent that interaction directly.
- Predictors are standardized separately inside each cross-validation training fold. This is important for Elastic Net because its regularization depends on predictor scale. The final fitted model is trained on all rows after the search, so its reported in-sample metrics are descriptive and are not an independent estimate of forecasting performance.
- The fitness function adds a fixed feature-count penalty of 0.1 to an MSE measured in squared throughput units. This is the specified heuristic objective, not a dimensionless or scale-invariant criterion; its practical influence depends on the scale of the response.
- The example does not model time ordering, changing relationships, missing data, or operational constraints. Its five-fold cross-validation treats rows as exchangeable observations and is suitable here as a demonstration of the stated method, not as a complete time-series forecasting evaluation.

### How the genetic algorithm works

A chromosome represents one candidate solution. Its first \(p\) genes are binary indicators: a value of 1 includes a predictor and 0 excludes it. The remaining genes are the ridge-scale parameter, the lasso-scale parameter, and the Elastic Net mixing ratio. A population is a collection of such candidates. Selection favors lower-fitness candidates; crossover recombines parent genes; mutation changes genes; and elitism carries the best candidates into the next generation.

For a chromosome \(z\), let \(S(z)\) be its selected predictor set, let \(n_{\mathrm{features}}(z)=|S(z)|\), and let \(k\) index the five cross-validation folds. The fitness minimized by the GA is

$$\
F(z)=\frac{1}{5}\sum_{k=1}^{5}\operatorname{MSE}_k(z)+0.1\,n_{\mathrm{features}}(z).
$$

Here, \(\operatorname{MSE}_k(z)\) is the held-out mean squared error for fold \(k\), using the selected predictors and the Elastic Net model encoded by \(z\). The coefficient 0.1 is the fixed feature penalty used in this example. A chromosome with no selected predictors receives a large penalty and is not fitted.

The two parameter genes \(a_r\) and \(a_l\) are combined with the mixing gene \(\rho\) as

$$\
\alpha=a_r(1-\rho)+a_l\rho,\qquad l1\_ratio=\rho,\qquad 0\leq\rho\leq1.
$$

Thus the estimator is Elastic Net, with \(\alpha\) controlling the overall penalty and \(l1\_ratio\) controlling the L1-versus-L2 mixture. The parameter genes are a hybrid encoding used by this search; they do not mean that three separate regression estimators are fitted.

If \(P\) is the population size, \(G\) the number of generations, and \(K=5\) the number of folds, fitness evaluation requires approximately \(GPK\) model fits, in addition to the final fit. Exhaustive feature-subset enumeration alone would involve up to \(2^p\) subsets, before considering continuous parameter values. The GA searches a sample of this mixed discrete-and-continuous space rather than enumerating it or certifying an optimum.

### What to examine

- The best-fitness curve should be non-increasing because the best candidate found so far is retained; the average population fitness need not be monotonic.
- The selected-feature display and the synthetic signal/noise labels show how the search behaved on this generated dataset. They are not evidence of causal discovery.
- The selected chromosome's cross-validation objective and the final model's in-sample fit measure different things and should not be treated as interchangeable performance estimates.
- Genetic algorithms are stochastic. A fixed seed makes this notebook's sequence reproducible in a given software environment, but does not make the method a guarantee of global optimality.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# A fixed seed makes the synthetic data and GA's NumPy-based operations reproducible.
np.random.seed(42)

n_samples = 120

# These six variables enter the response-generating equation.
trade_volume = np.random.normal(100, 15, n_samples)
gdp_growth = np.random.normal(2.5, 0.8, n_samples)
fuel_price = np.random.normal(85, 10, n_samples)
seasonal_peak = np.random.binomial(1, 0.3, n_samples)
ship_density = np.random.normal(25, 5, n_samples)
exchange_rate = np.random.normal(1.2, 0.15, n_samples)

# These six variables are generated independently of the response equation.
random_noise_1 = np.random.normal(0, 1, n_samples)
random_noise_2 = np.random.normal(0, 1, n_samples)
random_noise_3 = np.random.normal(0, 1, n_samples)
competition = np.random.normal(50, 10, n_samples)
weather_index = np.random.normal(100, 20, n_samples)
policy_dummy = np.random.binomial(1, 0.2, n_samples)

# The interaction contributes to the response but is not supplied as a candidate feature.
true_throughput = (
    2000
    + 15 * trade_volume
    + 300 * gdp_growth
    + 8 * fuel_price
    + 250 * seasonal_peak
    + 20 * ship_density
    + 150 * exchange_rate
    + 50 * (trade_volume * gdp_growth) / 100
    + np.random.normal(0, 250, n_samples)
)

data = pd.DataFrame(
    {
        "trade_volume": trade_volume,
        "gdp_growth": gdp_growth,
        "fuel_price": fuel_price,
        "seasonal_peak": seasonal_peak,
        "ship_density": ship_density,
        "exchange_rate": exchange_rate,
        "random_noise_1": random_noise_1,
        "random_noise_2": random_noise_2,
        "random_noise_3": random_noise_3,
        "competition": competition,
        "weather_index": weather_index,
        "policy_dummy": policy_dummy,
        "throughput": true_throughput,
    }
)

predictor_columns = [column for column in data.columns if column != "throughput"]

print("Synthetic port-throughput dataset")
print(f"Shape: {data.shape}")
print(f"Number of candidate predictors: {len(predictor_columns)}")
print("\nPredictor variables:")
for index, column in enumerate(predictor_columns, start=1):
    print(f"  {index:2d}. {column}")

print("\nDataset summary statistics:")
print(data.describe().round(2))

In [ ]:
class GeneticAlgorithmRegressor:
    """Use a genetic algorithm to select features and tune Elastic Net parameters."""

    def __init__(
        self,
        n_predictors,
        population_size=40,
        crossover_rate=0.8,
        mutation_rate=0.15,
        elite_size=4,
        max_generations=100,
    ):
        self.n_predictors = n_predictors
        self.population_size = population_size
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.max_generations = max_generations

        # Each chromosome contains one binary gene per predictor and three real-valued genes.
        self.chromosome_length = n_predictors + 3

    def initialize_population(self):
        """Create a diverse initial population of feature masks and parameter values."""
        population = []

        for _ in range(self.population_size):
            binary_genes = np.random.binomial(1, 0.5, self.n_predictors)
            ridge_alpha = np.random.uniform(0.01, 10.0)
            lasso_alpha = np.random.uniform(0.01, 10.0)
            elastic_ratio = np.random.uniform(0.0, 1.0)

            chromosome = np.concatenate(
                [binary_genes, [ridge_alpha, lasso_alpha, elastic_ratio]]
            )
            population.append(chromosome)

        return np.array(population)

    def evaluate_fitness(self, chromosome, X, y):
        """Return five-fold cross-validated MSE plus the feature-count penalty."""
        feature_mask = chromosome[: self.n_predictors].astype(bool)
        ridge_alpha = chromosome[self.n_predictors]
        lasso_alpha = chromosome[self.n_predictors + 1]
        elastic_ratio = chromosome[self.n_predictors + 2]

        if not np.any(feature_mask):
            return 1e6

        X_selected = X[:, feature_mask]

        try:
            # The two scale genes are blended into the Elastic Net alpha parameter.
            alpha = ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio
            model = make_pipeline(
                StandardScaler(),
                ElasticNet(
                    alpha=alpha,
                    l1_ratio=elastic_ratio,
                    max_iter=1000,
                    random_state=42,
                ),
            )

            # Scaling is fitted within each training fold to avoid validation-fold leakage.
            cv_scores = cross_val_score(
                model,
                X_selected,
                y,
                cv=5,
                scoring="neg_mean_squared_error",
            )
            mse = -cv_scores.mean()
            complexity_penalty = 0.1 * np.sum(feature_mask)
            return mse + complexity_penalty

        except (ValueError, FloatingPointError):
            return 1e6

    def tournament_selection(self, population, fitness_values, tournament_size=3):
        """Select parents by repeatedly choosing the lowest-fitness tournament member."""
        selected = []

        for _ in range(len(population)):
            tournament_indices = np.random.choice(
                len(population), tournament_size, replace=False
            )
            winner_idx = tournament_indices[
                np.argmin(fitness_values[tournament_indices])
            ]
            selected.append(population[winner_idx].copy())

        return np.array(selected)

    def crossover(self, parent1, parent2):
        """Recombine binary genes by uniform exchange and real genes by blending."""
        if np.random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()

        child1 = parent1.copy()
        child2 = parent2.copy()

        for index in range(len(parent1)):
            if index < self.n_predictors:
                if np.random.random() < 0.5:
                    child1[index], child2[index] = parent2[index], parent1[index]
            elif np.random.random() < 0.5:
                blend = np.random.random()
                child1[index] = blend * parent1[index] + (1 - blend) * parent2[index]
                child2[index] = (1 - blend) * parent1[index] + blend * parent2[index]

        return child1, child2

    def mutate(self, chromosome):
        """Flip selected binary genes or perturb real-valued genes within their bounds."""
        mutated = chromosome.copy()

        for index in range(len(chromosome)):
            if np.random.random() < self.mutation_rate:
                if index < self.n_predictors:
                    mutated[index] = 1 - mutated[index]
                elif index == self.n_predictors:
                    mutated[index] = np.clip(
                        mutated[index] + np.random.normal(0, 1), 0.01, 10.0
                    )
                elif index == self.n_predictors + 1:
                    mutated[index] = np.clip(
                        mutated[index] + np.random.normal(0, 1), 0.01, 10.0
                    )
                else:
                    mutated[index] = np.clip(
                        mutated[index] + np.random.normal(0, 0.2), 0.0, 1.0
                    )

        return mutated

    def evolve(self, X, y, verbose=True):
        """Run initialization, evaluation, selection, crossover, mutation, and elitism."""
        if X.ndim != 2 or X.shape[1] != self.n_predictors:
            raise ValueError("X must have one column per configured predictor.")
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of observations.")
        if self.elite_size < 0 or self.elite_size > self.population_size:
            raise ValueError("elite_size must be between zero and population_size.")

        population = self.initialize_population()
        best_fitness_history = []
        avg_fitness_history = []
        fitness_dispersion_history = []
        best_chromosome = None
        best_fitness = float("inf")

        if verbose:
            print("Genetic algorithm evolution progress:")
            print("=" * 60)

        for generation in range(self.max_generations):
            fitness_values = np.array(
                [self.evaluate_fitness(chromosome, X, y) for chromosome in population]
            )

            current_best_idx = np.argmin(fitness_values)
            current_best_fitness = fitness_values[current_best_idx]

            if current_best_fitness < best_fitness:
                best_fitness = current_best_fitness
                best_chromosome = population[current_best_idx].copy()

            best_fitness_history.append(best_fitness)
            avg_fitness_history.append(np.mean(fitness_values))
            fitness_dispersion_history.append(np.std(fitness_values))

            if verbose and (
                generation == 0
                or generation % 20 == 19
                or generation == self.max_generations - 1
            ):
                feature_mask = best_chromosome[: self.n_predictors].astype(bool)
                selected_indices = np.flatnonzero(feature_mask)
                ridge_alpha = best_chromosome[self.n_predictors]
                elastic_ratio = best_chromosome[self.n_predictors + 2]

                print(
                    f"Generation {generation}: best fitness = {best_fitness:.4f}"
                )
                print(f"  Selected predictor indices: {selected_indices.tolist()}")
                print(
                    f"  Ridge-scale gene: {ridge_alpha:.4f}, "
                    f"Elastic Net ratio: {elastic_ratio:.4f}"
                )

            selected_population = self.tournament_selection(population, fitness_values)
            new_population = []

            # Elitism preserves the best current candidates in the next generation.
            elite_indices = np.argsort(fitness_values)[: self.elite_size]
            for index in elite_indices:
                new_population.append(population[index].copy())

            while len(new_population) < self.population_size:
                parent_indices = np.random.choice(
                    len(selected_population), 2, replace=False
                )
                parent1, parent2 = selected_population[parent_indices]
                child1, child2 = self.crossover(parent1, parent2)
                new_population.extend(
                    [self.mutate(child1), self.mutate(child2)]
                )

            population = np.array(new_population[: self.population_size])

        return {
            "best_chromosome": best_chromosome,
            "best_fitness": best_fitness,
            "fitness_history": best_fitness_history,
            "avg_fitness_history": avg_fitness_history,
            "fitness_dispersion_history": fitness_dispersion_history,
        }

In [ ]:
X = data[predictor_columns].to_numpy()
y = data["throughput"].to_numpy()

print("Data prepared for the genetic algorithm:")
print(f"Samples: {X.shape[0]}")
print(f"Predictors: {X.shape[1]}")
print(
    f"Chromosome length: {X.shape[1] + 3} "
    f"({X.shape[1]} binary genes and 3 real-valued genes)"
)

ga = GeneticAlgorithmRegressor(
    n_predictors=len(predictor_columns),
    population_size=40,
    crossover_rate=0.8,
    mutation_rate=0.15,
    elite_size=4,
    max_generations=100,
)

ga_results = ga.evolve(X, y, verbose=True)

In [ ]:
best_chromosome = ga_results["best_chromosome"]
best_fitness = ga_results["best_fitness"]
feature_mask = best_chromosome[: len(predictor_columns)].astype(bool)
selected_features = [
    feature for feature, selected in zip(predictor_columns, feature_mask) if selected
]
ridge_alpha = best_chromosome[len(predictor_columns)]
lasso_alpha = best_chromosome[len(predictor_columns) + 1]
elastic_ratio = best_chromosome[len(predictor_columns) + 2]

print("\n" + "=" * 60)
print("GENETIC ALGORITHM RESULTS")
print("=" * 60)
print(f"\nSelected features: {selected_features}")
print(f"Number of selected features: {len(selected_features)}/{len(predictor_columns)}")
print(f"Ridge-scale gene: {ridge_alpha:.4f}")
print(f"Lasso-scale gene: {lasso_alpha:.4f}")
print(f"Elastic Net ratio: {elastic_ratio:.4f}")
print(f"Blended Elastic Net alpha: {ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio:.4f}")
print(f"Best cross-validated fitness: {best_fitness:.4f}")

X_selected = X[:, feature_mask]
final_alpha = ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio
final_model = make_pipeline(
    StandardScaler(),
    ElasticNet(
        alpha=final_alpha,
        l1_ratio=elastic_ratio,
        max_iter=1000,
        random_state=42,
    ),
)
final_model.fit(X_selected, y)

# These metrics describe the model fitted to all rows; they are not held-out estimates.
y_pred = final_model.predict(X_selected)
final_mse = mean_squared_error(y, y_pred)
final_r2 = r2_score(y, y_pred)
final_estimator = final_model.named_steps["elasticnet"]

print("\nFinal model in-sample fit:")
print(f"Training MSE: {final_mse:.2f}")
print(f"Training R-squared: {final_r2:.4f}")
print("Coefficients on standardized predictors:")
for feature, coefficient in zip(selected_features, final_estimator.coef_):
    print(f"  {feature}: {coefficient:.4f}")
print(f"Intercept: {final_estimator.intercept_:.4f}")

# Check that the reported chromosome and fitted model use the same nonempty feature set.
assert np.any(feature_mask)
assert len(selected_features) == len(final_estimator.coef_)
assert np.isfinite(best_fitness)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Genetic Algorithm for Elastic Net Model Selection",
    fontsize=16,
    fontweight="bold",
)

# The best-so-far curve is monotonic under elitist tracking; the population mean can vary.
generations = np.arange(len(ga_results["fitness_history"]))
axes[0, 0].plot(
    generations,
    ga_results["fitness_history"],
    linewidth=2,
    label="Best fitness found so far",
)
axes[0, 0].plot(
    generations,
    ga_results["avg_fitness_history"],
    linestyle="--",
    linewidth=2,
    label="Mean population fitness",
)
axes[0, 0].set_xlabel("Generation")
axes[0, 0].set_ylabel("Fitness (cross-validated MSE + feature penalty)")
axes[0, 0].set_title("Fitness across generations")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Display every candidate predictor so inclusion and exclusion are both visible.
selection_values = feature_mask.astype(int)
selection_colors = ["tab:green" if selected else "tab:gray" for selected in selection_values]
axes[0, 1].bar(
    np.arange(len(predictor_columns)),
    selection_values,
    color=selection_colors,
    edgecolor="black",
)
axes[0, 1].set_xticks(np.arange(len(predictor_columns)))
axes[0, 1].set_xticklabels(predictor_columns, rotation=55, ha="right")
axes[0, 1].set_yticks([0, 1], labels=["Excluded", "Selected"])
axes[0, 1].set_ylim(-0.15, 1.25)
axes[0, 1].set_ylabel("Feature-selection decision")
axes[0, 1].set_title("Selected feature subset")
axes[0, 1].grid(axis="y", alpha=0.3)

# The color scale represents the mixing gene, while the axes show both scale genes.
parameter_plot = axes[1, 0].scatter(
    [ridge_alpha],
    [lasso_alpha],
    c=[elastic_ratio],
    cmap="viridis",
    vmin=0,
    vmax=1,
    s=220,
    edgecolors="black",
)
axes[1, 0].set_xlabel("Ridge-scale gene")
axes[1, 0].set_ylabel("Lasso-scale gene")
axes[1, 0].set_title("Selected chromosome parameters")
axes[1, 0].set_xlim(0, 10)
axes[1, 0].set_ylim(0, 10)
axes[1, 0].grid(True, alpha=0.3)
colorbar = fig.colorbar(parameter_plot, ax=axes[1, 0])
colorbar.set_label("Elastic Net ratio")

# This is an in-sample diagnostic, not a test-set forecast evaluation.
axes[1, 1].scatter(y, y_pred, alpha=0.7, edgecolors="none")
plot_limits = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
axes[1, 1].plot(plot_limits, plot_limits, "k--", label="Perfect agreement")
axes[1, 1].set_xlim(plot_limits)
axes[1, 1].set_ylim(plot_limits)
axes[1, 1].set_xlabel("Observed throughput (synthetic units)")
axes[1, 1].set_ylabel("Fitted throughput (synthetic units)")
axes[1, 1].set_title("In-sample fitted versus observed values")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("FEATURE-SELECTION AND SEARCH DIAGNOSTICS")
print("=" * 60)

# These labels are known from the synthetic data-generation equation, not inferred causal facts.
signal_generating_features = [
    "trade_volume",
    "gdp_growth",
    "fuel_price",
    "seasonal_peak",
    "ship_density",
    "exchange_rate",
]
noise_features = [
    "random_noise_1",
    "random_noise_2",
    "random_noise_3",
    "competition",
    "weather_index",
    "policy_dummy",
]

print("Signal-generating variables:")
for feature in signal_generating_features:
    status = "Selected" if feature in selected_features else "Not selected"
    print(f"  {feature}: {status}")

print("\nGenerated noise variables:")
for feature in noise_features:
    status = "Excluded" if feature not in selected_features else "Selected"
    print(f"  {feature}: {status}")

true_positives = sum(
    feature in selected_features for feature in signal_generating_features
)
false_positives = sum(feature in selected_features for feature in noise_features)
false_negatives = len(signal_generating_features) - true_positives
true_negatives = len(noise_features) - false_positives

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives
    else 0.0
)
recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives
    else 0.0
)
accuracy = (true_positives + true_negatives) / len(predictor_columns)

print("\nAgreement with the synthetic feature labels:")
print(f"  Precision: {precision:.3f}")
print(f"  Recall: {recall:.3f}")
print(f"  Accuracy: {accuracy:.3f}")
print("  These metrics assess agreement with the construction labels, not causal validity.")

fitness_history = ga_results["fitness_history"]
if len(fitness_history) >= 20:
    reference_fitness = fitness_history[-20]
    final_fitness = fitness_history[-1]
    if np.isclose(reference_fitness, 0.0):
        recent_improvement = 0.0
    else:
        recent_improvement = (reference_fitness - final_fitness) / abs(reference_fitness)

    print("\nBest-so-far fitness change over the most recent 20-generation interval:")
    print(f"  Relative improvement: {recent_improvement * 100:.2f}%")
    if recent_improvement < 0.01:
        print("  The best-so-far objective changed by less than 1% over this interval.")
    else:
        print("  The best-so-far objective improved by at least 1% over this interval.")

recent_dispersion = ga_results["fitness_dispersion_history"][-10:]
print(
    "Population fitness standard deviation in the final 10 generations "
    f"(mean): {np.mean(recent_dispersion):.4f}"
)
print(
    "Fitness dispersion summarizes objective-value spread; it is not a direct "
    "measure of chromosome or feature-subset diversity."
)

In [ ]:
print("\n" + "=" * 60)
print("OBJECTIVE AND MODEL-FIT DIAGNOSTICS")
print("=" * 60)

# The best fitness is the five-fold CV MSE plus the specified feature-count penalty.
feature_penalty = 0.1 * len(selected_features)
implied_cv_mse = best_fitness - feature_penalty

print(f"Selected feature count: {len(selected_features)}")
print(f"Feature-count penalty: {feature_penalty:.4f}")
print(f"Implied five-fold cross-validated MSE: {implied_cv_mse:.4f}")
print(f"Best penalized cross-validation fitness: {best_fitness:.4f}")
print(f"Final all-data training MSE: {final_mse:.4f}")
print(f"Final all-data training R-squared: {final_r2:.4f}")

assert np.isclose(best_fitness, implied_cv_mse + feature_penalty)
print(
    "\nThe cross-validated objective and all-data training metrics use different "
    "evaluation conditions; interpret them separately."
)

## Interpreting the method

### Why use a genetic algorithm for this search?

Feature inclusion is discrete, while the parameter genes are continuous. The GA represents both kinds of decisions in one chromosome and evaluates the resulting model with the same cross-validation objective. Its population-based search can explore multiple candidate regions, but it is stochastic and does not certify a global optimum. Elitism preserves the best candidate found so far; it does not prove that no better candidate exists.

### Strengths and limitations

**Strengths**

- Searches feature subsets and Elastic Net parameter values within one evolutionary process.
- Uses cross-validation error as its predictive component and includes an explicit preference for smaller feature sets.
- Represents binary feature decisions and continuous parameter values in a single hybrid chromosome.

**Limitations**

- Repeated cross-validation makes the search computationally expensive as population size, generation count, or dataset size grows.
- Results can depend on the random search path and GA settings; a low observed fitness is not a proof of global optimality.
- The fixed feature penalty is scale-dependent, and this synthetic example does not provide an independent time-ordered test set.
- Selected predictors describe the fitted predictive model. They do not, by themselves, establish causal effects.

### When this approach is useful

This approach is useful when a model-selection problem combines a large discrete feature-subset space with continuous tuning parameters and the computational budget permits repeated model fitting. The chromosome, fitness function, and evolutionary operators should be interpreted together: the chromosome defines candidate decisions, cross-validation and the complexity term score them, and selection, crossover, mutation, and elitism guide the search.